# 01 - Data Generation (DineIQ Analytics)

Generates the raw synthetic restaurant datasets with the SRS minimums:

| Dataset | SRS minimum |
| --- | --- |
| Order lines | 1,000,000 |
| Orders | 100,000 |
| Customers | 50,000 |
| Menu items / categories / locations | 150 / 10 / 20 |
| Months of history | 12 |
| Ratings | 100,000 |
| Wastage records | 50,000 |

**Relationship guarantees** (enforced in the generator):
- every order line uses the order's own restaurant menu,
- promotion references are active on the order date at the order's restaurant,
- ratings reference items actually present in the rated order,
- pricing / inventory / wastage reference valid item-restaurant pairs.

**Deliberate difficult cases** are injected at small documented rates
(missing values, duplicates, invalid lines, price outliers, inconsistent
totals, orphan ratings, out-of-period orders) so the cleaning stage has
real work to detect and handle.


In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import nb_common

# ------------------------------------------------------------------
# Run mode:
#   "quick" -> demo-scale data (~20k orders) in notebook/outputs/
#   "full"  -> full SRS scale (1M lines) in the canonical repo dirs
# ------------------------------------------------------------------
MODE = "quick"

paths = nb_common.setup_paths(MODE)
nb_common.ensure_config(paths)
mk = nb_common.markers(paths)

pd.options.display.max_columns = 20
print(f"MODE: {MODE}")
print(f"raw       : {paths['raw']}")
print(f"processed : {paths['processed']}")


MODE: quick
raw       : /home/user/Techwizz/notebook/outputs/quick/raw
processed : /home/user/Techwizz/notebook/outputs/quick/processed


In [2]:
from generate_dineiq_data import generate

if nb_common.step_done(*mk['generation']):
    print('Raw data already generated - skipping (delete', paths['raw'], 'to force a rerun).')
    counts = {n.stem: len(pd.read_csv(paths['raw'] / n)) for n in paths['raw'].glob('*.csv')}
else:
    counts = generate(raw_dir=paths['raw'], config_path=paths['config'])

Raw data already generated - skipping (delete /home/user/Techwizz/notebook/outputs/quick/raw to force a rerun).


In [3]:
pd.Series({k: v for k, v in counts.items() if isinstance(v, (int, float))}).sort_values(ascending=False)

order_items        200040
orders              20000
ratings             20000
inventory           10000
wastage             10000
customers            8040
pricing_history       272
menu_items            150
promotions            120
locations              20
restaurants            20
menu_categories        10
dtype: int64

In [4]:
from pathlib import Path

raw = paths['raw']
for name in ['orders.csv', 'order_items.csv', 'promotions.csv', 'ratings.csv']:
    df = pd.read_csv(raw / name, low_memory=False, nrows=5)
    print(f'--- {name} (first 5 rows) ---')
    display(df)

--- orders.csv (first 5 rows) ---


,order_id,customer_id,restaurant_id,order_date,order_time,order_status,order_channel,payment_method,promotion_id,subtotal,discount_amount,tax_amount,delivery_fee,total_amount
0,1,5366,6,2025-05-25,13:14:00,Cancelled,Takeaway,Cash,94.0,17245.58,2126.75,755.94,0,15874.77
1,2,4198,3,2025-07-14,17:37:00,Completed,Website/App,Online Wallet,NaN,10075.18,0.00,503.76,0,10578.94
2,3,7106,2,2025-06-10,20:52:00,Completed,Dine-in,Online Wallet,NaN,16577.12,2486.57,704.53,0,14795.08
3,4,4499,10,2025-06-09,20:23:00,Cancelled,Third-party Delivery,Card,NaN,4278.88,427.89,192.55,150,4193.54
4,5,211,18,2025-04-26,13:59:00,Completed,Takeaway,Online Wallet,NaN,7136.42,0.00,356.82,0,7493.24


--- order_items.csv (first 5 rows) ---


,order_item_id,order_id,menu_item_id,quantity,unit_price,discount_amount,line_total
0,1,1,86,2,757.91,0.00,1515.82
1,2,1,146,1,1145.62,0.00,1145.62
2,3,1,26,1,1102.53,0.00,1102.53
3,4,1,46,2,1340.65,402.19,2681.30
4,5,1,6,2,788.75,0.00,1577.50


--- promotions.csv (first 5 rows) ---


,promotion_id,promotion_name,promotion_type,menu_item_id,category_id,restaurant_id,start_date,end_date,discount_percentage,minimum_order_value,usage_limit
0,1,Promotion 001,Seasonal,78,8,18,2025-02-01,2025-03-28,5,0,1000
1,2,Promotion 002,Bundle,88,8,8,2025-08-01,2025-09-28,5,1500,1000
2,3,Promotion 003,Bundle,17,7,17,2025-05-01,2025-06-28,10,1000,1000
3,4,Promotion 004,Bundle,114,4,14,2025-10-01,2025-10-28,20,500,5000
4,5,Promotion 005,Seasonal,107,7,7,2025-02-01,2025-04-28,25,0,500


--- ratings.csv (first 5 rows) ---


,rating_id,customer_id,order_id,menu_item_id,restaurant_id,rating,review_date,review_text
0,1,4618,11798,71,11,5,2025-01-18,Excellent
1,2,2699,17229,71,11,4,2025-06-03,Could be better
2,3,1913,6713,68,8,4,2025-06-24,Good value
3,4,7395,5176,80,20,3,2025-03-18,Good experience
4,5,79,1338,39,19,5,2025-04-04,Could be better


In [5]:
# Verify the three relationship guarantees on the RAW data
from pathlib import Path
raw = paths['raw']
od = pd.read_csv(raw / 'orders.csv', low_memory=False)
oi = pd.read_csv(raw / 'order_items.csv', low_memory=False)
mi = pd.read_csv(raw / 'menu_items.csv', low_memory=False)
ra = pd.read_csv(raw / 'ratings.csv', low_memory=False)

# 1. every order line belongs to the order's own restaurant
item_rest = mi.set_index('menu_item_id')['restaurant_id']
m = oi.merge(od[['order_id', 'restaurant_id']], on='order_id')
mismatch = int((m['menu_item_id'].map(item_rest) != m['restaurant_id']).sum())
print('1. order-line restaurant mismatches (must be 0):', mismatch)

# 2. ratings: item must be in the order (small injected orphan fraction expected)
lines = oi.groupby('order_id')['menu_item_id'].apply(set).to_dict()
orphan = int(sum(1 for r in ra.itertuples() if r.menu_item_id not in lines.get(r.order_id, set())))
print(f'2. orphan ratings (intentionally injected, expect ~0.5%): {orphan} / {len(ra)}')

# 3. out-of-period orders (intentionally injected, expect ~0.1%)
oob = int((~od['order_date'].astype(str).between('2025-01-01', '2025-12-31')).sum())
print(f'3. out-of-period orders (intentionally injected): {oob} / {len(od)}')

1. order-line restaurant mismatches (must be 0): 0


2. orphan ratings (intentionally injected, expect ~0.5%): 100 / 20000
3. out-of-period orders (intentionally injected): 20 / 20000


**Next:** `02_data_quality_check.ipynb` - assess the raw data before cleaning.
